## 1. Environment Setup

Initialize the observability dependencies and configure Phoenix and Langfuse connections.

This section prepares the notebook for tracing, monitoring, and evaluating the SEVA_AI RAG pipeline.

In [1]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv

from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

from langfuse import get_client

load_dotenv()

PROJECT_NAME = "SEVA_AI"

PHOENIX_ENDPOINT = "http://localhost:6006/v1/traces"

LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_HOST = os.getenv("LANGFUSE_HOST", "http://localhost:3000")

print("Environment loaded")
print("Phoenix endpoint:", PHOENIX_ENDPOINT)
print("Langfuse host:", LANGFUSE_HOST)
print("Langfuse credentials available:",
      bool(LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY))

Environment loaded
Phoenix endpoint: http://localhost:6006/v1/traces
Langfuse host: http://localhost:3000
Langfuse credentials available: True


## 2. Phoenix Integration

Initialize Phoenix OpenTelemetry tracing and connect the LangChain instrumentation to the Docker-hosted Phoenix server.

In [2]:
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

tracer_provider = register(
    endpoint=PHOENIX_ENDPOINT,
    project_name=PROJECT_NAME
)

LangChainInstrumentor().instrument(
    tracer_provider=tracer_provider
)

print("Phoenix tracing initialized successfully.")

OpenTelemetry Tracing Details
|  Phoenix Project: SEVA_AI
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phoenix tracing initialized successfully.


## 3. Phoenix Test Trace

Create a sample OpenTelemetry span and verify that it is successfully exported to the Phoenix server.

In [3]:
from opentelemetry import trace

tracer = trace.get_tracer("seva_ai.observability")

with tracer.start_as_current_span("seva_ai_test_trace") as span:
    span.set_attribute("project", "SEVA_AI")
    span.set_attribute("experiment", "observability_setup")
    span.set_attribute("retrieval_method", "hybrid_rrf_cohere")

print("Test trace created and exported.")

Test trace created and exported.


## 4. Manual RAG Pipeline Tracing

Create a reusable OpenTelemetry tracing helper for instrumenting individual RAG stages. Each stage will appear as a child span under the main query trace.

In [4]:
from contextlib import contextmanager
from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode
import time

tracer = trace.get_tracer("seva_ai.rag")

@contextmanager
def trace_stage(stage_name, attributes=None):
    start_time = time.perf_counter()

    with tracer.start_as_current_span(stage_name) as span:
        if attributes:
            for key, value in attributes.items():
                span.set_attribute(key, value)

        try:
            yield span

        except Exception as e:
            span.record_exception(e)
            span.set_status(Status(StatusCode.ERROR, str(e)))
            raise

        finally:
            elapsed = time.perf_counter() - start_time
            span.set_attribute("stage.duration_seconds", elapsed)

### Test Manual Tracing

Create a parent RAG query span and one child retrieval span to verify the tracing hierarchy.

In [5]:
with tracer.start_as_current_span("SEVA_AI_RAG_QUERY") as parent:
    parent.set_attribute("project", "SEVA_AI")

    with trace_stage(
        "dense_retrieval",
        {"retrieval.method": "dense", "retrieval.top_k": 30}
    ):
        print("Dense retrieval stage traced")

Dense retrieval stage traced


## Phoenix — Custom Tracer for SEVA_AI

Create a reusable OpenTelemetry tracer for custom spans around the existing retrieval pipeline.

In [10]:
from opentelemetry import trace

tracer = trace.get_tracer(
    "seva_ai.retrieval",
    "1.0.0"
)

print("Tracer initialized:", tracer)

Tracer initialized: <opentelemetry.sdk.trace.Tracer object at 0x0000022BDBB7C050>


## Phoenix — Dense Retrieval Instrumentation

Instrument the existing Dense retrieval function with a custom Phoenix span. Capture query details, retrieval configuration, chunk count, unique scheme count, top-ranked scheme IDs, execution latency, and exceptions without changing retrieval logic.

In [14]:
from pathlib import Path

MODEL_PATH = Path(
    r"G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2"
)

required_files = [
    "config.json",
    "model.safetensors",
    "modules.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.txt",
    "1_Pooling/config.json"
]

missing_files = [
    file for file in required_files
    if not (MODEL_PATH / file).exists()
]

if not missing_files:
    print("Model files verified successfully!")
    print("Model location:", MODEL_PATH)
else:
    print("Missing files:", missing_files)

Model files verified successfully!
Model location: G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2


In [15]:
import os
import time

os.environ["HF_HUB_OFFLINE"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings

start_time = time.perf_counter()

embedding_model = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

load_time = time.perf_counter() - start_time

test_embedding = embedding_model.embed_query(
    "What government schemes are available for farmers?"
)

print(f"Model loading time: {load_time:.2f} seconds")
print("Embedding dimensions:", len(test_embedding))
print("Model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loading time: 214.69 seconds
Embedding dimensions: 384
Model loaded successfully!


In [16]:
import json
import time
from pathlib import Path
from collections import defaultdict

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

VECTORSTORE_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\vectorstores")

discovery_db = Chroma(
    collection_name="seva_discovery_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "discovery")
)

detail_db = Chroma(
    collection_name="seva_detail_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "detail")
)

discovery_count = discovery_db._collection.count()
detail_count = detail_db._collection.count()

print("Discovery collection:", discovery_count)
print("Detail collection:", detail_count)
print("Total indexed chunks:", discovery_count + detail_count)

Discovery collection: 23438
Detail collection: 67945
Total indexed chunks: 91383


In [17]:
import time
import json

from opentelemetry.trace import Status, StatusCode

# Retrieval configuration
RETRIEVAL_K = 30

def retrieve_schemes(query: str, k: int = RETRIEVAL_K):

    with tracer.start_as_current_span("dense_retrieval") as span:

        start = time.perf_counter()

        try:
            # Input attributes
            span.set_attribute("retrieval.method", "dense")
            span.set_attribute("retrieval.query", query)
            span.set_attribute("retrieval.top_k", k)

            results = discovery_db.similarity_search_with_score(
                query=query,
                k=k
            )

            latency = time.perf_counter() - start

            # Deduplicate chunks into ranked schemes
            ranked_schemes = []
            seen_scheme_ids = set()

            for rank, (doc, score) in enumerate(results, start=1):
                scheme_id = doc.metadata.get("scheme_id")

                if scheme_id and scheme_id not in seen_scheme_ids:
                    seen_scheme_ids.add(scheme_id)

                    ranked_schemes.append({
                        "scheme_id": scheme_id,
                        "rank": len(ranked_schemes) + 1,
                        "chunk_rank": rank,
                        "distance": float(score),
                        "section": doc.metadata.get("child_type")
                    })

            # Output attributes
            span.set_attribute("retrieval.chunk_count", len(results))
            span.set_attribute(
                "retrieval.unique_scheme_count",
                len(ranked_schemes)
            )
            span.set_attribute(
                "retrieval.latency_seconds",
                latency
            )
            span.set_attribute(
                "retrieval.top_scheme_ids",
                json.dumps([
                    scheme["scheme_id"]
                    for scheme in ranked_schemes[:10]
                ])
            )

            span.set_status(Status(StatusCode.OK))

            return {
                "query": query,
                "retrieved_chunks": len(results),
                "retrieved_schemes": ranked_schemes,
                "latency": latency
            }

        except Exception as exc:
            span.record_exception(exc)
            span.set_status(
                Status(StatusCode.ERROR, str(exc))
            )
            raise

In [18]:
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np

# Project directories
PROJECT_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai")

GOLDEN_PATH = (
    PROJECT_DIR
    / "notebooks/golden_dataset/discovery/seva_discovery_golden_v1.jsonl"
)

DISCOVERY_DB_PATH = Path(r"G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\vectorstores\discovery")

In [19]:
golden_df = pd.read_json(GOLDEN_PATH, lines=True)

test_query = golden_df.iloc[0]["question"]

test_result = retrieve_schemes(test_query)

print("Query:", test_query)
print("Retrieved chunks:", test_result["retrieved_chunks"])
print("Unique schemes:", len(test_result["retrieved_schemes"]))
print("Latency:", round(test_result["latency"], 3), "seconds")

for scheme in test_result["retrieved_schemes"][:10]:
    print(
        f"Rank: {scheme['rank']} | "
        f"Scheme ID: {scheme['scheme_id']} | "
        f"Chunk Rank: {scheme['chunk_rank']}"
    )

Query: I need a loan to purchase a light motor vehicle for my transport business.
Retrieved chunks: 30
Unique schemes: 19
Latency: 0.714 seconds
Rank: 1 | Scheme ID: ZaL-yJoBXu-iUKXqmUCT | Chunk Rank: 1
Rank: 2 | Scheme ID: GMZdU5kB0AGc0F0ifzmY | Chunk Rank: 2
Rank: 3 | Scheme ID: hqIa0pwBXu-iUKXq00o3 | Chunk Rank: 3
Rank: 4 | Scheme ID: 3he_F5gB5uF80_MGg5Hz | Chunk Rank: 6
Rank: 5 | Scheme ID: ksaOaZsB0AGc0F0ilf7I | Chunk Rank: 7
Rank: 6 | Scheme ID: dca3F5gB0AGc0F0iMjSR | Chunk Rank: 8
Rank: 7 | Scheme ID: oKLGz5oBXu-iUKXqxUAg | Chunk Rank: 10
Rank: 8 | Scheme ID: mcb3T5sB0AGc0F0iTP2L | Chunk Rank: 13
Rank: 9 | Scheme ID: 6aJOaZsBXu-iUKXqSUKJ | Chunk Rank: 14
Rank: 10 | Scheme ID: hca_2ZoB0AGc0F0iCPyl | Chunk Rank: 15
